# Chapter 16 &mdash; Graph Colouring as a Boolean Formula

**Concept 17 of the Chapter 16 decomposition:** *Graph Colouring as a Boolean Formula*

Two bits per region, one constraint per border &mdash; and the diagram returns every proper colouring at once.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Graph-Colouring-As-A-Formula/Concept-Graph-Colouring-As-A-Formula.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Four-colouring is NP-complete, and the reduction to SAT is the kind a programmer
actually writes. Give each region **two Boolean variables**, so its colour is a two-bit
number; for each pair of adjacent regions, assert the two bit-pairs are **not equal**.

Nothing else is needed. There is no "every region gets a colour" constraint, because two
bits always denote one of four colours &mdash; the encoding enforces it for free, which
is what a good encoding does.

In Jove's spec language a border reads directly:

```
Nv_not_Ut = ~((aNV <=> aUT) & (bNV <=> bUT))
```

The map is Utah, Nevada, Arizona and Colorado. Their adjacency graph is $K_4$ with one
edge missing (Nevada and Colorado do not touch), and the BDD returns not *a* colouring
but **all** of them &mdash; a number the chromatic polynomial can confirm independently.

The negative case teaches as much: squeeze a triangle into **one** bit per region and
the diagram collapses to the bare `0` node.

## 2. Definitions

### The map, as a formula

In [ ]:
FOURCOL = '''
Var_Order : aUT bUT aNV bNV aAZ bAZ aCO bCO

Nv_not_Ut = ~((aNV <=> aUT) & (bNV <=> bUT))
Nv_not_Az = ~((aNV <=> aAZ) & (bNV <=> bAZ))
Az_not_Ut = ~((aUT <=> aAZ) & (bUT <=> bAZ))
Co_not_Az = ~((aCO <=> aAZ) & (bCO <=> bAZ))
Co_not_Ut = ~((aCO <=> aUT) & (bCO <=> bUT))

Main_Exp : Nv_not_Ut & Nv_not_Az & Az_not_Ut & Co_not_Az & Co_not_Ut
'''

def colour_of(model, state):
    # the two bits of a state, read as a number 0..3
    return 2 * model['a' + state] + model['b' + state]

print(FOURCOL)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;16.&nbsp;Counting Solutions: a BDD Does #SAT in One Pass](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Counting-Solutions-Sharp-SAT/Concept-Counting-Solutions-Sharp-SAT.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16-NPC/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;18.&nbsp;Variable Ordering, and What BDDs Do Not Settle](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Variable-Ordering-And-P-Versus-NP/Concept-Variable-Ordering-And-P-Versus-NP.ipynb)&nbsp;&rarr;

---

## 3. Tests

Build it. Every satisfying assignment is a proper colouring.

In [ ]:
g = bdd(FOURCOL)
g.report()
g

**Check the count against a closed form.** $K_4$ minus an edge has chromatic polynomial $k(k-1)(k-2)^2$, which at $k=4$ is 48.

In [ ]:
print('BDD says                   :', g.count, 'colourings')
print('k(k-1)(k-2)^2 at k = 4     :', 4 * 3 * 2 * 2)
assert g.count == 4 * 3 * 2 * 2
print()
print('An independent formula agreeing with the diagram is worth more than')
print('any number of spot checks.')

Read a few colourings back out, and verify every border by hand.

In [ ]:
BORDERS = [('UT','NV'), ('UT','AZ'), ('UT','CO'), ('NV','AZ'), ('AZ','CO')]
STATES  = ['UT', 'NV', 'AZ', 'CO']

for m in g.models[:5]:
    print('   ' + '  '.join('%s=%d' % (s, colour_of(m, s)) for s in STATES))

for m in g.models:
    for x, y in BORDERS:
        assert colour_of(m, x) != colour_of(m, y), (m, x, y)
print()
print('All %d colourings checked: no two bordering states share a colour.'
      % g.count)

**The unsatisfiable side.** A triangle with one bit per region &mdash; two colours &mdash; cannot be coloured, and the diagram says so by being the `0` node.

In [ ]:
tri = bdd('''
Var_Order : x y z
Main_Exp : (x XOR y) & (y XOR z) & (x XOR z)
''')
print(repr(tri))
print('2-colourable?', tri.is_sat)
assert not tri.is_sat
print()
print('A triangle needs 3 colours, and one bit offers 2.  The encoding')
print('could not be satisfied, so the reduced diagram is a single 0.')

## 4. Exercises


1. Add California (bordering Nevada and Arizona) and re-count. Does the chromatic
   polynomial still apply, and if not, why not?
2. Give the triangle **two** bits per region and re-run. How many colourings, and
   does $k(k-1)(k-2)$ agree?
3. The encoding has no "every region gets a colour" constraint. Write the three-colour
   version, where two bits allow a *fourth* value that must be excluded, and say what
   that costs.
4. Nevada and Colorado do not share a border. Add the constraint anyway and re-count.
   Which chromatic polynomial does the new number match?
5. Turn the colouring into a CNF with `cnf()` and count its clauses. Would you want
   to hand that to a SAT solver, or the original five constraints?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16-NPC/Concept-Graph-Colouring-As-A-Formula')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')